**Preliminari**

In [1]:
import sys
import os
from dotenv import load_dotenv
from pathlib import Path

# if notebook is in PRIN/notebooks, parent() is PRIN
project_root = Path.cwd().resolve().parent
sys.path.insert(0, str(project_root))
print("Added to sys.path:", project_root)

import json
from utils.schema_json import ReportData, AnnotatedReport
import time
from IPython.display import clear_output

from huggingface_hub import login

import torch
import torch.nn as nn

from transformers import AutoTokenizer, AutoModel
from datasets import load_dataset, Dataset, DatasetDict

Added to sys.path: C:\Users\lucat\PycharmProjects\PRIN


**Impostiamo il device, scheda video se disponibile**

In [2]:
print(f'{torch.cuda.is_available() = }')  # True se la GPU è disponibile
print(f'{torch.cuda.device_count() = }')  # Numero di GPU disponibili
print(f'{torch.cuda.get_device_name(0) = }')  # Nome della GPU

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f'{device = }')

torch.cuda.is_available() = True
torch.cuda.device_count() = 1
torch.cuda.get_device_name(0) = 'NVIDIA GeForce RTX 5070'
device = device(type='cuda')


**Huggingface login**

In [3]:
# Set the API key for HuggingFace
load_dotenv()  # Load environment variables from .env file
hf_api_key = os.getenv("HF_TOKEN")
login(token=hf_api_key)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


**Parametri**

In [4]:
# Parameters
TRAIN_FILE_NAME = "data_finetune_guido_openai_train.jsonl"
VALIDATION_FILE_NAME = "data_finetune_guido_openai_val.jsonl"
#FILE_NAMES = (TRAIN_FILE_NAME, VALIDATION_FILE_NAME)
TIPO = 'openai'

CHECKPOINT = "bert-base-multilingual-cased"

# Fields we don't want to predict
EXCLUDED_FIELDS = (
    'ore_inizio',
    'ore_fine',
    'spessore_parietale',
    'estensione_cranio_caudale',
    'distanza_oai',
    'linfonodi_sospetti',
    'numero_depositi'
)

**Load data**

In [5]:
# Carichiamo i nostri file JSON
file_names = {
    'train': TRAIN_FILE_NAME,
    'validation': VALIDATION_FILE_NAME,
}

paths = {
    split: Path('../data/ft-dataset/' + file_name) for split, file_name in file_names.items()
}

data = dict()
for split, path in paths.items():
    with open(path, "r", encoding="utf-8") as f:
        data_list = [json.loads(line) for line in f]
        data[split] = data_list

train_data, validation_data = data['train'], data['validation']

print(f"{len(train_data) = }")
print(f"{type(train_data[0]) = }")
print(f"{type(train_data[0]['messages'][1]['content']) = }")  # Report text
print(f"{type(train_data[0]['messages'][2]['content']) = }")  # Annotations

len(train_data) = 116
type(train_data[0]) = <class 'dict'>
type(train_data[0]['messages'][1]['content']) = <class 'str'>
type(train_data[0]['messages'][2]['content']) = <class 'str'>


In [6]:
annotated_reports: dict[str, list[AnnotatedReport]] = {split: [] for split in file_names.keys()}
for split in annotated_reports:
    for record in data[split]:
        report_text = record['messages'][1]['content'].lower()  # Tutte lettere minuscole
        if TIPO == 'openai':
            report_data = ReportData.model_validate_json(record['messages'][2]['content'])
        else:
            report_data = ReportData.model_validate(record['messages'][2]['content'])
        annotated_reports[split].append(AnnotatedReport(report_text=report_text, report_data=report_data))

**Load model and tokenizer**

In [7]:
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)
model = AutoModel.from_pretrained(CHECKPOINT).to(device)

In [8]:
print(f"{tokenizer.pad_token_type_id = }")

tokenizer.pad_token_type_id = 0


In [9]:
# Check the maximum number of tokens for each report
max_n_tokens_train = 0
del_train = []
for i, report in enumerate(annotated_reports['train']):
    x = tokenizer(report.report_text, return_tensors='pt')['input_ids'].shape[1]
    max_n_tokens_train = max(max_n_tokens_train, x)
    if x > model.config.max_position_embeddings:
        del_train.append(i)
print(del_train)
print(f'{max_n_tokens_train = }')

# Check the maximum number of tokens for each report
max_n_tokens_validation = 0
del_val = []
for i, report in enumerate(annotated_reports['validation']):
    x = tokenizer(report.report_text, return_tensors='pt')['input_ids'].shape[1]
    max_n_tokens_validation = max(max_n_tokens_validation, x)
    if x > model.config.max_position_embeddings:
        del_val.append(i)
print(del_val)
print(f'{max_n_tokens_validation = }')

# Delete long reports
for i in del_train[::-1]:
    annotated_reports['train'].pop(i)
for i in del_val[::-1]:
    annotated_reports['validation'].pop(i)
print('deleted')

Token indices sequence length is longer than the specified maximum sequence length for this model (659 > 512). Running this sequence through the model will result in indexing errors


[86, 95, 97, 111]
max_n_tokens_train = 659
[1, 5, 17, 26]
max_n_tokens_validation = 949
deleted


In [10]:
def create_hugging_face_dataset(annotated_reports: list[AnnotatedReport]) -> Dataset:
    text = []
    for report in annotated_reports:
        text.append(report.report_text)
    return Dataset.from_dict({'text': text})

In [11]:
dataset = DatasetDict({
    'train': create_hugging_face_dataset(annotated_reports['train']),
    'validation': create_hugging_face_dataset(annotated_reports['validation'])
})

In [12]:
def tokenize_function(examples, padding="max_length", max_length=model.config.max_position_embeddings):
    return tokenizer(examples['text'], padding=padding, max_length=max_length)

In [13]:
dataset = dataset.map(tokenize_function, batched=True)
dataset.set_format('torch')
print(dataset)

Map:   0%|          | 0/112 [00:00<?, ? examples/s]

Map:   0%|          | 0/24 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 112
    })
    validation: Dataset({
        features: ['text', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 24
    })
})


**Test without attention**

In [14]:
input = dataset['train'][0:2]['input_ids'].to(device)
print(input)

tensor([[   101,  10294, 106687,  ...,      0,      0,      0],
        [   101,  10106, 109822,  ...,      0,      0,      0]],
       device='cuda:0')


In [15]:
output = model(input)
print(output['pooler_output'])

We strongly recommend passing in an `attention_mask` since your input_ids may be padded. See https://huggingface.co/docs/transformers/troubleshooting#incorrect-output-when-padding-tokens-arent-masked.


tensor([[ 0.2992, -0.1450,  0.2331,  ..., -0.3839,  0.2487,  0.2114],
        [ 0.2205, -0.1004,  0.1621,  ..., -0.2607,  0.1066,  0.1246]],
       device='cuda:0', grad_fn=<TanhBackward0>)


**Test with attention**

In [16]:
input_attention = dataset['train'][0:2]['attention_mask'].to(device)
print(input_attention)

tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]], device='cuda:0')


In [17]:
output_attention = model(input, attention_mask=input_attention)
print(output_attention['pooler_output'])

tensor([[ 0.2265, -0.0852,  0.1328,  ..., -0.3184,  0.1100,  0.1947],
        [-0.0736, -0.3641,  0.1564,  ..., -0.4149,  0.1317,  0.2486]],
       device='cuda:0', grad_fn=<TanhBackward0>)


**Test with original text no padding (-> no attention needed)**

Results are the same of those obtained with padding + attention

In [18]:
print(model(torch.tensor(tokenizer(dataset['train'][0:1]['text'])['input_ids']).to(device))['pooler_output'][0][:5])
print(model(torch.tensor(tokenizer(dataset['train'][1:2]['text'])['input_ids']).to(device))['pooler_output'][0][:5])

tensor([ 0.2266, -0.0852,  0.1328, -0.1539, -0.0256], device='cuda:0',
       grad_fn=<SliceBackward0>)
tensor([-0.0736, -0.3641,  0.1564, -0.4477, -0.3163], device='cuda:0',
       grad_fn=<SliceBackward0>)


# Test model to predict only morfologia and infiltrazione tessuto adiposo

In [20]:
class ReportExtractor(nn.Module):
    def __init__(self, bert_checkpoint=CHECKPOINT, num_classes_morfologia=4, num_classes_tessuto_adiposo=5):
        super().__init__()
        self.bert = AutoModel.from_pretrained(bert_checkpoint)
        hidden = self.bert.config.hidden_size  # Embedding length
        self.dropout = nn.Dropout(0.2)
        # Classification heads
        # Morfologia head
        self.class_morfologia = nn.Linear(hidden, num_classes_morfologia)
        # Tessuto adiposo head
        self.class_tessuto_adiposo = nn.Linear(hidden, num_classes_tessuto_adiposo)

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        
        # Better than pooler_output
        cls = out.last_hidden_state[:, 0, :]
        # Alternative:
        #cls = out.pooler_output
        cls = self.dropout(cls)
    
        return {
            "morfologia": self.class_morfologia(cls),
            "tessuto_adiposo": self.class_tessuto_adiposo(cls),
        }

In [29]:
# Example
report_extractor = ReportExtractor().to(device)
report_extractor.eval()
report_extractor(dataset['train'][0:1]['input_ids'].to(device), attention_mask=dataset['train'][0:1]['attention_mask'].to(device))

{'morfologia': tensor([[ 0.0525, -0.0133, -0.0495, -0.1008]], device='cuda:0',
        grad_fn=<AddmmBackward0>),
 'tessuto_adiposo': tensor([[0.1706, 0.0163, 0.1297, 0.2668, 0.1404]], device='cuda:0',
        grad_fn=<AddmmBackward0>)}

In [30]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 112
    })
    validation: Dataset({
        features: ['text', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 24
    })
})